In [1]:
import scCube
from scCube import scCube
from scCube.visualization import *
from scCube.utils import *
from pathlib import Path
from matplotlib.pyplot import rc_context
import pandas as pd
import scanpy as sc
import numpy as np
import warnings
import time
warnings.filterwarnings("ignore")

In [2]:
model = scCube()

In [ ]:
train_sample = '151669'
sc_data = pd.read_csv(f"./DLPFC/{train_sample}/data.csv", index_col = 0)
sc_meta = pd.read_csv(f"./DLPFC/{train_sample}/meta.csv", index_col = 0)
sc_adata = model.pre_process(
    sc_data=sc_data.T, 
    sc_meta=sc_meta,
    is_normalized=False
    )
coor_df = sc_adata.obs[['imagerow','imagecol']]
sc_adata.obsm['spatial'] = coor_df.to_numpy()
sc_adata.obs['spot'] = sc_adata.obs.index
sc_adata.layers['data'] = sc_adata.X
sc.pl.embedding(sc_adata, basis='spatial', color='ground_truth', title='151669 raw', size=50, show=False)    

In [ ]:
selected_mask = np.full(len(sc_adata.obs), False)
cond1 = (sc_adata.obs['array_col'] >= 87) & (sc_adata.obs['array_col'] <= 90) & (sc_adata.obs['array_row'] < 3)
cond2 = (sc_adata.obs['array_col'] > 90) & (sc_adata.obs['array_col'] <= 95) & (sc_adata.obs['array_row'] < 4)
cond3 = (sc_adata.obs['array_col'] > 95) & (sc_adata.obs['array_col'] <= 110) & (sc_adata.obs['array_row'] < 7)
selected_mask = cond1 | cond2 | cond3
selected_cells = sc_adata.obs[selected_mask]
cells_to_remove = selected_cells.index
sc_adata = sc_adata[~sc_adata.obs_names.isin(cells_to_remove)].copy()
sc.pl.embedding(sc_adata, basis='spatial', color='ground_truth', title='151669 raw', size=50, show=False)  
obs_df = sc_adata[sc_adata.obs['ground_truth'] == 'Layer3'].obs.copy()
selected_ids = []
for col_val, group in obs_df.groupby('array_col'):
    sorted_group = group.sort_values(by='array_row')
    top_n = sorted_group.head(3)
    selected_ids.extend(top_n.index)
selected_df = obs_df.loc[selected_ids]
max_row_indices = selected_df.sort_values('array_row', ascending=False).head(12).index
selected_ids = [idx for idx in selected_ids if idx not in max_row_indices]
sc_adata.obs['ground_truth'] = sc_adata.obs['ground_truth'].cat.add_categories(['Layer2'])
sc_adata.obs.loc[selected_ids, 'ground_truth'] = 'Layer2'
sc.pl.embedding(sc_adata, basis='spatial', color='ground_truth', title='151669 new', size=50, show=False)    

In [ ]:
for train_sample in ['151673', '151674', '151675', '151676']:
    sp_data = pd.read_csv(f"./DLPFC/{train_sample}/data.csv", index_col=0)
    sp_meta = pd.read_csv(f"./DLPFC/{train_sample}/meta.csv", index_col=0)
    sp_adata = model.pre_process(
        sc_data=sp_data.T, 
        sc_meta=sp_meta,
        is_normalized=False
        )
    sp_adata.obs['spot'] = sp_adata.obs.index
    sp_adata.layers['data'] = sp_adata.X
    layer = 'Layer2'
    test_adata1 = sc_adata[sc_adata.obs['ground_truth'] == layer].copy()
    test_adata2 = sc_adata[sc_adata.obs['ground_truth'] != layer].copy()
    train_adata = sp_adata[sp_adata.obs['ground_truth'] == layer].copy()
    generate_sc_meta1, generate_sc_data1 = model.load_vae_and_generate_cell(
        sc_adata=train_adata,
        celltype_key='ground_truth',
        cell_key='spot',
        target_num=dict(test_adata1.obs.ground_truth.value_counts()),
        hidden_size=128,
        load_path=f"./train/DLPFC_{train_sample}.pth",
        used_device='cuda:0')
    generate_sc_data1, generate_sc_meta1 = model.generate_pattern_reference(
        sc_adata=test_adata1,
        generate_sc_data=generate_sc_data1,
        generate_sc_meta=generate_sc_meta1,
        celltype_key='ground_truth',
        spatial_key=['imagecol', 'imagerow'],
        cost_metric='sqeuclidean')
    generate_sc_data1.columns = [col + '_1' for col in generate_sc_data1.columns]
    generate_sc_meta1['Cell'] = generate_sc_meta1['Cell'].astype(str) + '_1'
    generate_sc_meta1 = generate_sc_meta1.set_index('Cell')
    common_cells1 = generate_sc_data1.columns.intersection(generate_sc_meta1.index)
    data_matched1 = generate_sc_data1[common_cells1]
    meta_matched1 = generate_sc_meta1.loc[common_cells1]
    new_adata1 = ad.AnnData(X=data_matched1.T, obs=meta_matched1, var=pd.DataFrame(index=data_matched1.index))

    generate_sc_meta2, generate_sc_data2 = model.load_vae_and_generate_cell(
        sc_adata=test_adata2,
        celltype_key='ground_truth',
        cell_key='spot',
        target_num=dict(test_adata2.obs.ground_truth.value_counts()),
        hidden_size=128,
        load_path=f"./train/DLPFC_{test_sample}.pth",
        used_device='cuda:0')
    generate_sc_data2, generate_sc_meta2 = model.generate_pattern_reference(
        sc_adata=test_adata2,
        generate_sc_data=generate_sc_data2,
        generate_sc_meta=generate_sc_meta2,
        celltype_key='ground_truth',
        spatial_key=['imagecol', 'imagerow'],
        cost_metric='sqeuclidean')
    generate_sc_data2.columns = [col + '_2' for col in generate_sc_data2.columns]
    generate_sc_meta2['Cell'] = generate_sc_meta2['Cell'].astype(str) + '_2'
    generate_sc_meta2 = generate_sc_meta2.set_index('Cell')
    common_cells2 = generate_sc_data2.columns.intersection(generate_sc_meta2.index)
    data_matched2 = generate_sc_data2[common_cells2]
    meta_matched2 = generate_sc_meta2.loc[common_cells2]
    new_adata2 = ad.AnnData(X=data_matched2.T, obs=meta_matched2, var=pd.DataFrame(index=data_matched2.index))

    new_adata = ad.concat([new_adata1, new_adata2])
    coor_df = new_adata.obs[['imagecol','imagerow']]
    new_adata.obsm['spatial'] = coor_df.to_numpy()
    new_adata.obs['ground_truth'] = new_adata.obs['Cell_type']
    OUTPUT_PATH = Path(f"..")
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    row_names = new_adata.obs.index.tolist()
    col_names = new_adata.var.index.tolist()
    df = pd.DataFrame(new_adata.X, index=row_names, columns=col_names)
    df.to_csv(f"{OUTPUT_PATH}/data.csv")
    new_adata.obs.to_csv(f"{OUTPUT_PATH}/meta.csv")
    new_adata.write_h5ad(f"{OUTPUT_PATH}/adata.h5ad")